In [37]:
import pandas as pd
import matplotlib.pyplot as plt

In [38]:
data=pd.read_csv("IMDB Dataset.csv")
data.head(50)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [39]:
data.shape

(50000, 2)

In [40]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [41]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [42]:
reviews = data["review"]
labels = data["sentiment"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    reviews,
    labels,
    test_size=0.2,
    random_state=42
)

In [44]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [45]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X_train)

In [46]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [47]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=200)
X_test_pad = pad_sequences(X_test_seq, maxlen=200)

In [48]:
# =========================
# 8. BUILD MODEL
# =========================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train_pad.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\tanay\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │        25,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,049 (133.00 KB)

 Trainable params: 34,049 (133.00 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
print(X_train_pad.shape)
print(X_test_pad.shape)

print(len(y_train))
print(len(y_test))

(40000, 200)
(10000, 200)
40000
10000


In [50]:
# train the model
history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,# train the model y_train,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.4996 - loss: 24.4142 - val_accuracy: 0.4965 - val_loss: 0.7860
Epoch 2/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5016 - loss: 0.7260 - val_accuracy: 0.4975 - val_loss: 0.7304
Epoch 3/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5002 - loss: 0.6990 - val_accuracy: 0.4950 - val_loss: 0.7174
Epoch 4/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5010 - loss: 0.6938 - val_accuracy: 0.4975 - val_loss: 0.7169
Epoch 5/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4985 - loss: 0.6964 - val_accuracy: 0.4970 - val_loss: 0.7144
Epoch 6/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5026 - loss: 0.6942 - val_accuracy: 0.4963 - val_loss: 0.7274
Epoch 7/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.5021 - loss: 0.6928 - val_accuracy: 0.4972 - val_loss: 0.7044
Epoch 8/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5001 - loss: 0.6953 -

In [51]:
# evaluate the model
loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=1)
print('Test Accuracy: ', accuracy)
print('Test Loss: ', loss)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step - accuracy: 0.4963 - loss: 0.6988
Test Accuracy:  0.49630001187324524
Test Loss:  0.6988108158111572


In [52]:
# classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))

Classification Report:


NameError: name 'classification_report' is not defined